# Primary Setup

In [13]:
import getpass
import os

def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Provide your {var}: ")

_set_if_undefined("GEMINI_API_KEY")
_set_if_undefined("TAVILY_API_KEY")

# All Installs

In [14]:
!uv pip install langgraph langchain_community langchain-tavily langchain_experimental langchain-google-genai

Checked 5 packages in 38ms



# All Imports

In [15]:

from typing import Annotated, List, Optional, Dict, Literal

import os

from langchain_community.document_loaders import WebBaseLoader
from langchain_core.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from typing_extensions import TypedDict

# Define the Tools

In [16]:
@tool("scrape_webpages", description="Scrape webpages and return their content as a string.")
def scrape_webpages(urls: List[str]) -> str:
    """Annotated function to scrape webpages and return their content as a string."""
    loader = WebBaseLoader(urls)
    docs = loader.load()
    return "\n\n".join(
        [
            f'<Document name="{doc.metadata.get("title", "")}">\n{doc.page_content}\n</Document>'
        ]
    )

@tool("create_outline", description="Create an outline for a document based on its content.")
def create_outline(
    points: Annotated[List[str], "A list of main points or sections."],
    file_name: Annotated[str, "The file path to save the outline."]
) -> Annotated[str, "The path of the saved outline file."]:
    """Annotated function to create and save an outline for a document based on its content."""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "w") as file:
        for i, point in enumerate(points):
            file.write(f"{i + 1}. {point}\n")
    return f"Outline saved to {file_name}"

@tool("read_document", description="Read a document and return its content.")
def read_document(
    file_name: Annotated[str, "The file path of the document to read."],
    start: Annotated[Optional[int], "The starting line number to read from."],
    end: Annotated[Optional[int], "The ending line number to read to."]
):
    """Annotated function to read a document and return its content."""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "r") as file:
        lines = file.readlines()
    if start is None:
        start = 0
    return "\n".join(lines[start:end])

@tool("write_document", description="Write a document and return its content.")
def write_document(
    content: Annotated[str, "The content to write to the document."],
    file_name: Annotated[str, "The file path to save the document."]
):
    """Annotated function to write a document and return its content."""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "w") as file:
        file.write(content)
    return f"Document saved to {file_name}"

@tool("edit_document", description="Edit a document and return its updated content.")
def edit_document(
    file_name: Annotated[str, "The file path of the document to edit."],
    insert: Annotated[Dict[int, str], "The dictionary where key is the line number and value is the text to insert."],
):
    """Annotated function to edit a document and return its updated content."""
    file_to_use = os.path.join(os.getcwd(), "temp", file_name)
    with open(file_to_use, "W") as file:
        lines = file.readlines()
    sorted_insert = sorted(insert.items())

    for line_number, text in sorted_insert:
        if 1 <= line_number <= len(lines) + 1:
            lines.insert(line_number - 1, text + "\n")
        else:
            return f"Error: Line number {line_number} is out of range. Document has {len(lines)} lines."

    with open(file_to_use, "w") as file:
        file.writelines(lines)

    return f"Document updated and saved to {file_name}"

# Define the Supervisor

In [21]:

class State(MessagesState):
    next: str

def make_supervisor_node(llm: BaseChatModel, members: List[str]) -> str:
    options = members + ["FINISH"]
    system_prompt = {
        "You are a supervisor tasked with managing the conversation between the"
        f" following workers: {members}. Given the following user request"
        " respond with the worker to act next. Each worker will perform a"
        " task and respond with their results and status. When finished,"
        " respond with 'FINISH'."
    }

    class Router(TypedDict):
        next: Literal[*Options]

    def supervisor_node(state: State) -> Command[Literal[*members, "__end__"]]:
        messages = [
            {"role": "system", "content": system_prompt},
        ] + state["messages"]

        response = llm.with_structured_output(Router).invoke(messages)
        goto = response["next"]
        if goto == "FINISH":
            return Command(goto=goto, update={"next": goto})

    return supervisor_node

# Define the Agent Teams

### 1. Research Team

In [34]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    api_key=os.environ.get("GEMINI_API_KEY")
)
tavily_tool = TavilySearch(max_results = 3)

search_agent = create_react_agent(llm, tools=[tavily_tool])

def search_node(state: State) -> Command[Literal["supervisor"]]:
    result = search_agent.invoke(state)
    return Command(
        update = {
            "messages": [HumanMessage(content=result["messages"][-1].content, name="search")]
        },
        goto = "supervisor"
    )

web_scrapper_agent = create_react_agent(llm, tools=[scrape_webpages])

def web_scrapper_node(state: State) -> Command[Literal["supervisor"]]:
    result = web_scrapper_agent.invoke(state)
    return Command(
        update = {
            "messages": [HumanMessage(content=result["messages"][-1].content, name="web_scrapper")]
        },
        goto = "supervisor"
    )

research_supervisor_node = make_supervisor_node(llm, ["search", "web_scrapper"])

/var/folders/q9/q040vc_d1xn_b5bb1lcwy8sw0000gn/T/ipykernel_4130/3976411439.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  search_agent = create_react_agent(llm, tools=[tavily_tool])
/var/folders/q9/q040vc_d1xn_b5bb1lcwy8sw0000gn/T/ipykernel_4130/3976411439.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  web_scrapper_agent = create_react_agent(llm, tools=[scrape_webpages])


In [35]:

research_builder = StateGraph(State)
research_builder.add_node("supervisor", research_supervisor_node)
research_builder.add_node("search", search_node)
research_builder.add_node("web_scrapper", web_scrapper_node)

research_builder.add_edge(START, "supervisor")
research_graph = research_builder.compile()

In [36]:
for s in research_graph.stream(
    {"messages" : [("user", "what is the weather like in Porto today?")]},
    {"recursion_limit": 20},
): 
    print(s)
    print("---")

GoogleInvalidRequestError: Error calling model 'gemini-3.6-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 'INVALID_ARGUMENT'}}

### 2. Writing Team

In [ ]:
doc_writer_agent = create_react_agent(
    llm, 
    tools = [write_document, edit_document, read_document],
    prompt = (
        "You can read, write and edit documents based on note taker's outlines. "
        "Don't ask follow up questions."
    )
)

def doc_writing_node(state: State) -> Command[Literal["supervisor"]]:
    result = doc_writer_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content = result["messages"][-1].content, name = "doc_writer")
            ]
        },
        goto = "supervisor",
    )

note_taking_agent = create_react_agent(
    llm,
    tools = [create_outline, read_document],
    prompt = (
        "You can read documents and create outlines for the document writer."
        "Don't ask follow up questions."
    )
)


def note_taking_node(state: State) -> Command[Literal["supervisor"]]:
    result = note_taking_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content = result["messages"][-1].content, name = "note_taker")
            ]
        },
        goto = "supervisor",
    )

chart_generating_agent = create_react_agent(
    llm, tools = [read_document, python_repl_tool]
)

def chart_generating_node(state: State) -> Command[Literal["supervisor"]]:
    result = chart_generating_agent.invoke(state)
    return Command(
        update = {
            "messages": [
                HumanMessage(content = result["messages"][-1].content, name = "chart_generator")
            ]
        },
        goto = "supervisor",
    )

doc_writing_supervisor_node = make_supervisor_node(
    llm, ["doc_writer", "note_taker", "chart_generator"]
)

In [ ]:
writing_builder = StateGraph(State)
writing_builder.add_node("supervisor", doc_writing_supervisor_node)
writing_builder.add_node("doc_writer", doc_writing_node)
writing_builder.add_node("note_taker", note_taking_node)
writing_builder.add_node("chart_generator", chart_generating_node)

writing_builder.add_edge(START, "supervisor")
writing_graph = writing_builder.compile()

In [ ]:
for s in writing_graph.stream(
    {
        "messages": [
            (
                "user",
                "Write an outline for a poem about dogs and after that write the poem itself and store it"
            )
  s      ]
    },
    {"recursion_limit": 30}
):
    print(s)
    print("---")

# End-to-End Graph

### Implement the graph

In [ ]:
teams_supervisor_node = make_supervisor_node(llm, ["research_team", "writing_team"])

def call_research_team(state: State) -> Command[Literal["supervisor"]]:
    response = research_graph.invoke({"messages": state["messages"][-1]})
    return Command(
        update = {
            "messages" : [
                HumanMessage(
                    content=response["messages"][-1].content, name = "research_team"
                )
            ]
        },
        goto="supervisor"
    )

def call_writing_team(state: State) -> Command[Literal["supervisor"]]:
    response = writing_graph.invoke({"messages": state["messages"][-1]})
    return Command(
        update = {
            "messages" : [
                HumanMessage(
                    content=response["messages"][-1].content, name = "writing_team"
                )
            ]
        },
        goto="supervisor"
    )

super_builder = StateGraph(State)
super_builder.add_node("supervisor", teams_supervisor_node)
super_builder.add_node("research_team", call_research_team)
super_builder.add_node("writing_team", call_writing_team)

super_builder.add_edge(START, "supervisor")
super_graph = super_builder.compile()

### Visualize Graph

In [ ]:
from IPython.display import Image, display

display(Image(super_graph.get_graph().draw_mermaid_png()))

### Run Graph

In [ ]:
for s in super_graph.stream(
    {
        "messages": [
            ("user", "Research why the gold price has been increasing crazily in 2025. Come up with the reasons and compile a report. Do not try to generate any charts or plots. Just a report with text and numbers will do.")
        ]
    },
    {"recursion_limit": 1000}
):
    print(s)
    print("....")